# Action Process Reward Models (Act-PRMs) — from scratch with 🤗 Transformers

*Companion notebook for* **[On Learning to Think with Action Process Reward Models](https://openreview.net/forum?id=2zsteCP2wy)** *(ICML 2026 Workshop on RL from World Feedback).*

**The idea in one paragraph.** We often have *action-only* demonstrations of successful workflows — process logs recording *what* a human (or agent) did (tool calls, replies), but never *why*. SFT on these logs alone underperforms: the explicit actions are only half the picture, missing the latent thoughts that connect them. Act-PRMs treat those thoughts as **latent variables**: we sample candidate thoughts $z$ from the LLM itself, score each by the likelihood it induces on the *observed next action*,

$$\tilde r(z) = p_\theta(x \mid s, z),$$

and train the LLM with policy gradient using the group-normalized reward $\bar r(z^{(g)}) = \tilde r(z^{(g)}) / \sum_{g'} \tilde r(z^{(g')})$ — which is exactly the **Expectation-Maximization** update for this latent-variable model.

**This notebook** implements every piece with plain Hugging Face Transformers + PyTorch, on a model small enough for a free Colab GPU:

1. **The data** — a tiny action-only trajectory
2. **The context gap** — why action-only prediction is hard (a 2-cell motivating study)
3. **Prompt reversal** — bootstrapping thought generation by showing the action *first*
4. **E-step** — sampling $G$ candidate thoughts
5. **The reward** — length-normalized action likelihood + EM normalization
6. **M-step** — a REINFORCE update
7. **The full loop** — and distilling relabelled traces back via SFT

> The paper's experiments use Qwen3-4B-Instruct-2507 + LoRA trained via [Tinker](https://thinkingmachines.ai/blog/announcing-tinker/); see the companion `act_prm_tinker.ipynb` for that route. Here we default to **Qwen3-0.6B** so everything runs on a T4.


## 0. Setup

In [ ]:
%pip install -q "transformers>=4.51" torch peft accelerate

In [ ]:
import math, textwrap, torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE}")

In [ ]:
# The paper uses "Qwen/Qwen3-4B-Instruct-2507"; 0.6B keeps this Colab-friendly.
MODEL_NAME = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
).to(DEVICE)
model.eval()
print(f"{MODEL_NAME}: {sum(p.numel() for p in model.parameters())/1e6:.0f}M params")

In [ ]:
def apply_template(messages, continue_final_message=False, add_generation_prompt=False):
    """Tokenize a chat with the model's template. `continue_final_message=True` leaves the
    final assistant message open (no EOS) so the model can be scored on / continue it."""
    kwargs = dict(
        tokenize=True,
        add_generation_prompt=add_generation_prompt,
        continue_final_message=continue_final_message,
    )
    try:  # Qwen3 templates accept this; other models may not
        return tokenizer.apply_chat_template(messages, enable_thinking=False, **kwargs)
    except (TypeError, ValueError):
        return tokenizer.apply_chat_template(messages, **kwargs)

## 1. The data: an action-only trajectory

A demonstration trajectory is a chat: observations arrive as `user`/`tool` messages, and each
`assistant` turn is an **explicit action** — a `<tool_call>` or a final answer. In a real pipeline these
come from process logs (the paper pulls them from HF datasets like
`mzio/aprm-snorkelai_agent_finance_reasoning`, filtered to successful rollouts). Note what's *missing*:
any reasoning between the actions.

The example below is an abridged step from the Snorkel Finance Reasoning task.

In [ ]:
SYSTEM_PROMPT = "You are a helpful assistant."

# Action-only trajectory: alternating (observation, explicit action) — no thoughts anywhere.
TRAJECTORY = [
    {"role": "user", "content": (
        "Here is the question: What is the company's lease financing strategy and how heavily "
        "does it rely on operating leases versus finance leases? "
        "The company to query in the database: meta"
    )},
    {"role": "assistant", "content":
        '<tool_call>{"name": "get_descriptions", "arguments": {"company_name": "meta"}}</tool_call>'},
    {"role": "user", "content": (
        "Result: 62 tables for 'meta', including "
        "'meta_LeaseBalanceSheetInformationTableTextBlock': lease assets and liabilities "
        "reported on the balance sheet."
    )},
    {"role": "assistant", "content":
        '<tool_call>{"name": "get_table_info", "arguments": {"company_name": "meta", '
        '"table_name": "meta_LeaseBalanceSheetInformationTableTextBlock"}}</tool_call>'},
]

# Index of the step we'll study: predict TRAJECTORY[3] given everything before it.
STATE_MESSAGES = [{"role": "system", "content": SYSTEM_PROMPT}] + TRAJECTORY[:3]
TARGET_ACTION = TRAJECTORY[3]["content"]
print(textwrap.fill("Target action: " + TARGET_ACTION, 100))

## 2. The reward: length-normalized action likelihood

Everything revolves around one function: **how likely is the logged action, given the state and a
candidate thought?** We compute it by teacher-forcing. Build the token sequence for
`state (+ thought) + action`, and the same sequence *without* the action; the difference in lengths
tells us which suffix tokens belong to the action; sum their log-probs and length-normalize:

$$\tilde r(z) \;=\; \exp\Big(\tfrac{1}{|x|}\textstyle\sum_{i \in x}\log p_\theta(\text{tok}_i \mid \text{tok}_{<i})\Big) \;\approx\; \text{PPL}(x \mid s, z)^{-1}$$

(This mirrors `compute_single_thought_action_metrics` in the companion codebase — including the
`continue_final_message=True` trick for the prefix.)

In [ ]:
@torch.no_grad()
def action_likelihood(state_messages, thought, target_action, return_details=False):
    """p(x | s, z): length-normalized probability of the action tokens.
    `thought=None` scores the action-only condition p(x | s)."""
    if thought is None:
        prefix_tokens = apply_template(state_messages, add_generation_prompt=True)
        full_msgs = state_messages + [{"role": "assistant", "content": target_action}]
    else:
        prefix_msgs = state_messages + [{"role": "assistant", "content": thought}]
        prefix_tokens = apply_template(prefix_msgs, continue_final_message=True)
        full_msgs = state_messages + [
            {"role": "assistant", "content": f"{thought}\n\n{target_action}"}
        ]
    full_tokens = apply_template(full_msgs)  # closed final message (EOS included)
    n_action = len(full_tokens) - len(prefix_tokens)

    ids = torch.tensor([full_tokens], device=DEVICE)
    logits = model(ids).logits.float()
    logprobs = F.log_softmax(logits[:, :-1], dim=-1)          # predicts tokens 1..N
    token_lp = logprobs.gather(-1, ids[:, 1:, None]).squeeze(-1)[0]
    action_lp = token_lp[-n_action:]                          # just the action suffix
    reward = action_lp.mean().exp().item()
    if return_details:
        return reward, action_lp
    return reward

### A 2-cell motivating study: the context gap

Before inferring thoughts, verify the paper's Section 2 premise on our single step: the action is
easier to predict when a connecting thought precedes it. (In the paper this gap is 9–14 accuracy
points across tasks, and SFT never closes it — Table 1.)

In [ ]:
hand_written_thought = (
    "The query result includes a table named 'meta_LeaseBalanceSheetInformationTableTextBlock', "
    "which likely contains information about Meta's lease financing strategy, including operating "
    "and finance leases. I will retrieve details from this specific table to answer the question."
)

r_action_only = action_likelihood(STATE_MESSAGES, None, TARGET_ACTION)
r_with_thought = action_likelihood(STATE_MESSAGES, hand_written_thought, TARGET_ACTION)

print(f"p(x | s)          (action-only)   = {r_action_only:.4f}")
print(f"p(x | s, z)       (with thought)  = {r_with_thought:.4f}")
print(f"ratio: {r_with_thought / r_action_only:.2f}x")

## 3. Prompt reversal: asking for the thought *behind* an action

How do we get useful thought samples from a model that was never trained to explain logged actions?
The bootstrap trick (`TinkerActionPromptActPrmGenerator` in the codebase): **show the action first**,
then prompt for the thought. Each assistant turn `{thought}<tool_call>{action}</tool_call>` is
re-ordered to

```
<tool_call>{action}</tool_call>

<thought>
{thought}
</thought>
```

and the final turn is truncated right after `<thought>\n`, so the model *continues* the message —
writing the thought for an action it can see. (The state-only Act-PRM generator later drops the hint;
same scoring either way.)

In [ ]:
THOUGHT_BOS, THOUGHT_EOS = "<thought>", "</thought>"
ACT_PRM_SYSTEM_PROMPT = (
    "You are a helpful assistant that infers reasoning thoughts behind your own observed actions."
)

def build_action_prompted_messages(state_messages, target_action):
    """(state, action) -> prompt that continues into the thought."""
    msgs = [{"role": "system", "content": ACT_PRM_SYSTEM_PROMPT}]
    msgs += [m for m in state_messages if m["role"] != "system"]
    msgs.append({"role": "assistant",
                 "content": f"{target_action}\n\n{THOUGHT_BOS}\n"})  # continue from here
    return msgs

reversal_msgs = build_action_prompted_messages(STATE_MESSAGES, TARGET_ACTION)
preview = tokenizer.apply_chat_template(reversal_msgs, tokenize=False, continue_final_message=True)
print(preview[-700:])

## 4. E-step: sample $G$ candidate thoughts

Sample with temperature 1.0 (the paper uses $G{=}8$; we use 4 to stay snappy), stopping at
`</thought>`.

In [ ]:
G = 4
MAX_THOUGHT_TOKENS = 96

@torch.no_grad()
def sample_thoughts(prompt_messages, g=G):
    prompt_tokens = apply_template(prompt_messages, continue_final_message=True)
    ids = torch.tensor([prompt_tokens] * g, device=DEVICE)
    out = model.generate(
        ids,
        do_sample=True, temperature=1.0, top_p=0.95,
        max_new_tokens=MAX_THOUGHT_TOKENS,
        pad_token_id=tokenizer.eos_token_id,
    )
    thoughts = []
    for row in out[:, ids.shape[1]:]:
        text = tokenizer.decode(row, skip_special_tokens=True)
        thoughts.append(text.split(THOUGHT_EOS)[0].strip())  # trim at closing tag
    return thoughts

thoughts = sample_thoughts(reversal_msgs)
for i, z in enumerate(thoughts):
    print(f"--- z^({i+1}) " + "-" * 60)
    print(textwrap.fill(z, 100), "\n")

## 5. Score + normalize: the Act-PRM reward

Score each thought with the *original* task prompt (reasoning-inference scaffolding stripped —
matching the codebase's `process_state_messages_for_metrics`), then group-normalize:

$$\bar r(z^{(g)}) = \frac{\tilde r(z^{(g)})}{\sum_{g'=1}^{G} \tilde r(z^{(g')})}$$

This is the empirical E-step posterior weight — and the per-sample reward for the policy gradient.

In [ ]:
rewards_raw = [action_likelihood(STATE_MESSAGES, z, TARGET_ACTION) for z in thoughts]
Z = sum(rewards_raw)
rewards = [r / Z for r in rewards_raw]
best = max(range(G), key=lambda g: rewards[g])

print(f"{'g':>3} {'~r (p(x|s,z))':>15} {'r-bar (EM)':>12}")
for g in range(G):
    tag = "  <- selected z-hat" if g == best else ""
    print(f"{g+1:>3} {rewards_raw[g]:>15.4f} {rewards[g]:>12.3f}{tag}")
print(f"\naction-only baseline p(x|s) = {r_action_only:.4f}")

## 6. M-step: a REINFORCE update

The EM M-step gradient is a vanilla policy gradient over the sampled thoughts:

$$\nabla_\theta J \approx \sum_{g=1}^{G} \bar r(z^{(g)})\, \nabla_\theta \log p_\theta(x, z^{(g)} \mid s)$$

So the loss we *minimize* is $-\sum_g \bar r(z^{(g)}) \log p_\theta(x, z^{(g)} \mid s)$, with the
state prefix masked out so gradient flows only through thought + action tokens. We wrap the model in
a LoRA adapter (as the paper does: $r{=}8$, $\alpha{=}16$ on the attention projections).

In [ ]:
from peft import LoraConfig, get_peft_model

lora_cfg = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()
optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=4e-5)

In [ ]:
def policy_gradient_step(state_messages, thoughts, rewards, target_action):
    """One M-step: weighted NLL over (thought + action) tokens, weights = EM rewards."""
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()
    for z, r_bar in zip(thoughts, rewards):
        prefix_tokens = apply_template(state_messages, add_generation_prompt=True)
        full_msgs = state_messages + [
            {"role": "assistant", "content": f"{z}\n\n{target_action}"}
        ]
        full_tokens = apply_template(full_msgs)
        n_gen = len(full_tokens) - len(prefix_tokens)   # thought + action tokens

        ids = torch.tensor([full_tokens], device=DEVICE)
        logits = model(ids).logits.float()
        logprobs = F.log_softmax(logits[:, :-1], dim=-1)
        token_lp = logprobs.gather(-1, ids[:, 1:, None]).squeeze(-1)[0]
        gen_lp = token_lp[-n_gen:].mean()               # length-normalized log-lik

        loss = -(r_bar * gen_lp) / len(thoughts)        # REINFORCE, reward detached by constr.
        loss.backward()
        total_loss += loss.item()
    torch.nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad), 1.0)
    optimizer.step()
    model.eval()
    return total_loss

loss = policy_gradient_step(STATE_MESSAGES, thoughts, rewards, TARGET_ACTION)
print(f"policy-gradient loss: {loss:.4f}")

## 7. The full loop

One **EM iteration** over a trajectory = walk its steps, sample+score+select thoughts at each
(E-step, committing the best thought $\hat z_t$ to context), then update on all of them (M-step).
Below is the whole algorithm in one compact function — Algorithm 1 of the paper.

In [ ]:
def em_iteration(trajectory, system_prompt=SYSTEM_PROMPT, g=G, verbose=True):
    """One EM iteration over an action-only trajectory. Returns relabelled trace."""
    state = [{"role": "system", "content": system_prompt}, trajectory[0]]
    relabelled, batch = [trajectory[0]], []

    action_indices = [i for i, m in enumerate(trajectory) if m["role"] == "assistant"]
    for t, idx in enumerate(action_indices):
        x_t = trajectory[idx]["content"]
        # E-step: sample candidates (action-prompted bootstrap), score, normalize
        zs = sample_thoughts(build_action_prompted_messages(state, x_t), g=g)
        raw = [action_likelihood(state, z, x_t) for z in zs]
        rbar = [r / sum(raw) for r in raw]
        z_hat = zs[max(range(g), key=lambda i: rbar[i])]
        if verbose:
            print(f"[t={t+1}] p(x|s,z-hat)={max(raw):.4f}  "
                  f"vs p(x|s)={action_likelihood(state, None, x_t):.4f}")
        batch.append((list(state), zs, rbar, x_t))
        # commit best thought + logged action, then next observation
        state.append({"role": "assistant", "content": f"{z_hat}\n\n{x_t}"})
        relabelled.append({"role": "assistant", "content": f"{z_hat}\n\n{x_t}"})
        if idx + 1 < len(trajectory):
            state.append(trajectory[idx + 1])
            relabelled.append(trajectory[idx + 1])

    # M-step over every step's sampled group
    for s_msgs, zs, rbar, x_t in batch:
        policy_gradient_step(s_msgs, zs, rbar, x_t)
    return relabelled

relabelled_trace = em_iteration(TRAJECTORY)

## 8. Distilling back: relabelled traces → SFT

After a few EM iterations, the trained model **relabels** each action-only log with its
highest-reward thoughts, producing a synthetic full-trajectory dataset $\{\hat\tau^{(i)}\}$. Standard
SFT on that dataset yields the final agent policy — in the paper this *matches or exceeds* SFT on
traces written by frontier LLMs (Claude Opus 4, o3, Gemini 2.5 Pro, …) on τ²-bench Airline and
Snorkel Finance, and beats action-only BC everywhere (+10 to +25 accuracy points).

In [ ]:
print("Relabelled trace (thoughts inferred, actions from the log):\n")
for m in relabelled_trace:
    body = textwrap.indent(textwrap.fill(m["content"], 96), "    ")
    print(f"[{m['role']}]\n{body}\n")

## Where to go next

- **`act_prm_tinker.ipynb`** — the same loop through the [Tinker](https://thinkingmachines.ai/blog/announcing-tinker/)
  training API (how the paper's runs actually executed): `sample_async` for the E-step,
  `compute_logprobs_async` for rewards, `forward_backward_async(loss_fn="importance_sampling")` for
  the M-step.
- **Companion codebase** (`act-prm-tinker`): the full multi-task pipeline — few-shot seeded
  environments (`environments/act_prm/env.py`), the two generators
  (`generator/tinker_act_prompt_aprm.py`, `generator/tinker_act_prm.py`), trainers with the
  distill-back stage (`trainer/act_prm.py`, `trainer/act_prm_sft_rl.py`).
- **The blog post** — interactive derivation and results.

```bibtex
@inproceedings{anonymous2026on,
  title={On Learning to Think with Action Process Reward Models},
  author={Michael Zhang and Madison Ho},
  booktitle={ICML 2026 Workshop on RL from World Feedback},
  year={2026},
  url={https://openreview.net/forum?id=2zsteCP2wy}
}
```